# 11 · Figuras de casos con ITCHI v0.3.0

Figuras del reporte de actividades (Evaluación Tutoral 7) a partir de `itchi.cases` (caché NetCDF de
`scripts/casos.py`). Convenciones de `itchi.plotting`: mapa base `configure_map_axis` / `MapStyle`, clases
SIAT con `plot_itchi_siat_levels` (umbrales de 0.2, `SIAT_DANGER_COLORS`), mapas de color de Crameri.

| Figura | Contenido |
|---|---|
| 0a | Cómo se calcula ITCHI en una posición: entradas, percentiles locales, regiones, $H_P$, $H_W$, ITCHI |
| 0b | Rampas: $H_P$ contra P6 (con R24 y A72), $H_W$ contra el viento y la unión de las dos |
| 1 | Máximo del evento en la malla, siete casos |
| 2 | Clase municipal del máximo del evento y municipios con declaratoria |
| 3 | $H_P$, $H_W$ e ITCHI en el pico del evento |
| 4 | Antes de los deslizamientos: ITCHI y $H_P$ municipales, y percentiles de P6, R24 y A72 |
| 5 | Evolución de ITCHI cada 6 h a lo largo de la trayectoria completa |
| Tablas | Clases por municipio y área; prueba ordinal con declaratorias |

In [ ]:
# 1. Configuración
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from itchi.cases import (CASES, DESLIZAMIENTOS, CaseConfig, class_counts, compute_case_fields,
                         declaratoria_inicio, experiment_settings, load_declaratorias, load_or_compute,
                         load_track, municipal_event, peak_time, percentile_level, precipitation_hazard,
                         read_climatologies, read_precip, t_event_utc, tabla_ordinal)
from itchi.constants import ITCHI_VERSION, SAFFIR_SIMPSON_HAZARD_NODES, TAIL_END_WIND_MS
from itchi.municipal import find_municipio, municipal_values, paint_municipal
from itchi.plotting import (MapStyle, SIAT_DANGER_COLORS, classify_itchi_to_siat_levels, configure_map_axis,
                            get_crameri_colormap, get_siat_hazard_cmap_norm, plot_itchi_field,
                            plot_itchi_siat_levels)

PC = ccrs.PlateCarree()
cfg = CaseConfig()
FIG_DIR = Path(cfg.fig_dir); FIG_DIR.mkdir(parents=True, exist_ok=True)
EXP = "itchi"                                   # experimento por defecto (configs/default.yaml)
MASCARA_TIERRA = True                           # figuras 1-2: ITCHI solo sobre tierra de México
FORMATOS = ("png", "pdf")                       # pdf para Overleaf (malla rasterizada)
ETIQUETAS = ("0 Sin peligro", "1 Muy bajo", "2 Bajo", "3 Moderado", "4 Alto", "5 Extremo")  # = SIAT_DANGER_LABELS
KT_POR_MS = 1.943844
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "savefig.dpi": 300})
print("itchi-core", ITCHI_VERSION, "| figuras en", FIG_DIR)

In [ ]:
# 2. Utilidades de figura
def guardar(fig, nombre):
    for ext in FORMATOS:
        fig.savefig(FIG_DIR / f"{nombre}.{ext}", bbox_inches="tight")
    print("guardada", nombre)


def titulo(case):
    return f"{CASES[case]['name'].title()} ({CASES[case]['season']})"


def categoria(vmax_kt):
    # Saffir-Simpson por viento máximo (kt)
    for lim, nom in ((34, "DT"), (64, "TT"), (83, "H1"), (96, "H2"), (113, "H3"), (137, "H4")):
        if vmax_kt < lim:
            return nom
    return "H5"


def cmap_crameri(nombre, reverse=False):
    try:
        return get_crameri_colormap(nombre, reverse=reverse)
    except AttributeError:                      # versión de cmcrameri sin ese mapa
        return get_crameri_colormap("batlow", reverse=reverse)


def sin_clase_cero(field, land=None):
    # clase 0 transparente (y fuera de tierra si se pide) para ver el mapa base
    f = np.where(np.asarray(field) > 0, field, np.nan)
    return np.where(land, f, np.nan) if land is not None else f


def estilo(lo, la, label_size=8.0):
    # MapStyle con marcas que dividen exactamente el recuadro (si no, set_xticks lo agranda)
    paso = 4.0 if max(lo[1] - lo[0], la[1] - la[0]) <= 24 else 8.0
    lo = (np.floor(lo[0] / paso) * paso, np.ceil(lo[1] / paso) * paso)
    la = (np.floor(la[0] / paso) * paso, np.ceil(la[1] / paso) * paso)
    return MapStyle(lon_extent=lo, lat_extent=la, lon_tick_spacing=paso, lat_tick_spacing=paso,
                    grid_spacing=paso / 2, label_size=label_size, coastline_linewidth=0.7, border_linewidth=0.6)


def recuadro(lon, lat, mask, margen=1.5):
    lon2d, lat2d = np.meshgrid(lon, lat)
    if not np.any(mask):
        return MapStyle().lon_extent, MapStyle().lat_extent
    return ((lon2d[mask].min() - margen, lon2d[mask].max() + margen),
            (lat2d[mask].min() - margen, lat2d[mask].max() + margen))


def barra_siat(fig, axes, **kw):
    # barra común de las clases 1-5 (la 0 es transparente en los mapas)
    cmap, norm = get_siat_hazard_cmap_norm()
    cb = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=axes, boundaries=np.arange(1, 7),
                      ticks=np.arange(1.5, 6.5), **kw)
    cb.set_ticklabels(ETIQUETAS[1:])
    cb.set_label("Clase de peligro ITCHI")
    return cb


def trayectoria(ax, track, usadas=None, color="k"):
    ax.plot(track["lon"], track["lat"], "-", color=color, lw=0.8, transform=PC, zorder=6)
    if usadas is not None:
        ax.plot(usadas[0], usadas[1], "o", ms=2.0, color="k", transform=PC, zorder=7)


def cuantiles(ds, horas):
    # {"Q75": malla, ...} de la climatología guardada en el caché
    pre = f"Q{horas}h_"
    return {v[len(pre):]: ds[v].values for v in ds.data_vars if v.startswith(pre)}


def con_huecos(times, values, paso_h=6):
    # serie para step(where="pre"): cada valor cubre las 6 h que terminan en su tiempo;
    # se inserta NaN donde falta el paso anterior (p. ej. sin posición del ciclón)
    t, v = pd.DatetimeIndex(times), np.asarray(values, float)
    dt = pd.Timedelta(hours=paso_h)
    tt, vv = [], []
    for i in range(len(t)):
        if i == 0 or t[i] - t[i - 1] != dt:
            tt.append(t[i] - dt); vv.append(np.nan)
        tt.append(t[i]); vv.append(v[i])
    return pd.DatetimeIndex(tt), np.array(vv)


def regiones(ax, lon, lat, m_dir, m_ind):
    # contorno de la región directa (continuo) y de toda la región de atribución (discontinuo)
    for m, ls in ((np.asarray(m_dir, float), "-"), (np.asarray(m_dir | m_ind, float), "--")):
        if m.any():
            ax.contour(lon, lat, m, levels=[0.5], colors="k", linewidths=0.7, linestyles=ls, transform=PC, zorder=6)

In [ ]:
# 3. Casos (lee el caché; calcula lo que falte)
CASOS = list(CASES)
C = {case: load_or_compute(case, cfg, verbose=True) for case in CASOS}
SET = experiment_settings(cfg)[EXP]
decl = load_declaratorias(cfg)
DECL = {case: set(declaratoria_inicio(decl, CASES[case]["atcf"]).index) for case in CASOS}
pd.DataFrame({case: dict(posiciones=C[case]["ds"].sizes["time"], trayectoria=len(C[case]["track"]),
                         municipios_con_declaratoria=len(DECL[case])) for case in CASOS}).T

## Figura 0a · Cómo se calcula ITCHI en una posición
Una posición de un caso (por defecto, el pico sobre México). Fila 1: entradas (lluvia de 6, 24 y 72 h de
MSWEP V3 y viento W06). Fila 2: nivel de percentil local de cada lluvia (climatología mayo–noviembre por
celda) y regiones de atribución (directa = R34, indirecta = hasta ROCLOUD con piso R_W06). Fila 3: $H_P$
(matriz P6 × R24 × A72), $H_W$ (Saffir-Simpson), ITCHI = unión de los dos (ver 0b) y su clase SIAT.

In [ ]:
CASO_ESQ, T_ESQ = "PATRICIA2015", "2015-10-23 18:00"   # None = pico sobre México; aquí, H5 antes de tocar tierra
Cc, ds = C[CASO_ESQ], C[CASO_ESQ]["ds"]
t = peak_time(Cc, EXP) if T_ESQ is None else pd.Timestamp(T_ESQ)
s, a = ds.sel(time=t, exp=EXP), ds.sel(time_all=t)
lon, lat = ds["lon"].values, ds["lat"].values
Q = {h: cuantiles(ds, h) for h in (6, 24, 72)}
lluvia = {"P6": a["P6"].values, "R24": a["R24"].values, "A72": a["A72"].values}
nivel = {k: percentile_level(lluvia[k], Q[h]) for k, h in (("P6", 6), ("R24", 24), ("A72", 72))}
m_dir, m_ind = s["M_direct"].values.astype(bool), s["M_indirect"].values.astype(bool)
lo, la = recuadro(lon, lat, m_dir | m_ind, margen=1.0)
st = estilo(lo, la, label_size=7.0)
centro = (float(s["track_lon"]), float(s["track_lat"]))

nombres_q = sorted(Q[6], key=lambda k: float(k[1:]))                 # p. ej. Q75, Q90, Q95, Q99
cmap_q = mcolors.ListedColormap(cmap_crameri("lajolla")(np.linspace(0.2, 0.9, len(nombres_q))))
norm_q = mcolors.BoundaryNorm(np.arange(-0.5, len(nombres_q)), cmap_q.N)
def a_indice(lv):                                                     # nivel 75/90/... -> 0, 1, ...
    out = np.full(lv.shape, np.nan)
    for i, k in enumerate(nombres_q):
        out[lv == float(k[1:])] = i
    return out

fig, axes = plt.subplots(3, 4, figsize=(16, 11), subplot_kw={"projection": PC}, layout="constrained")
letras = iter("abcdefghijkl")
def panel(ax, titulo_panel):
    configure_map_axis(ax, crs=PC, style=st)
    ax.plot(*centro, "*", ms=9, color="k", transform=PC, zorder=8)
    ax.set_title(f"({next(letras)}) {titulo_panel}", loc="left")

cm_ll = cmap_crameri("oslo", reverse=True)
for ax, k, nom in zip(axes[0, :3], ("P6", "R24", "A72"),
                      ("P6: lluvia en 6 h (mm)", "R24: lluvia en 24 h (mm)", "A72: 72 h antecedentes (mm)")):
    panel(ax, nom)
    art = ax.pcolormesh(lon, lat, np.where(lluvia[k] > 0.1, lluvia[k], np.nan), cmap=cm_ll, transform=PC,
                        norm=mcolors.LogNorm(1, max(10, np.nanpercentile(lluvia[k], 99.5))), rasterized=True, zorder=2)
    fig.colorbar(art, ax=ax, shrink=0.75)
panel(axes[0, 3], "Viento W06 (kt)")
w = s["wind_kt"].values
art = axes[0, 3].pcolormesh(lon, lat, np.where((m_dir | m_ind) & (w >= 1), w, np.nan),   # solo en la región de atribución
                            cmap=cmap_crameri("lapaz", reverse=True), transform=PC, rasterized=True, zorder=2)
cs = axes[0, 3].contour(lon, lat, w, levels=[TAIL_END_WIND_MS * KT_POR_MS, 34, 64], colors="k",
                        linewidths=[0.6, 0.9, 1.2], linestyles=[":", "-", "-"], transform=PC, zorder=7)
axes[0, 3].clabel(cs, fmt={cs.levels[0]: "12 m/s", cs.levels[1]: "34 kt", cs.levels[2]: "64 kt"}, fontsize=6)
fig.colorbar(art, ax=axes[0, 3], shrink=0.75)

for ax, k in zip(axes[1, :3], ("P6", "R24", "A72")):
    panel(ax, f"Percentil local de {k}")
    art = ax.pcolormesh(lon, lat, a_indice(nivel[k]), cmap=cmap_q, norm=norm_q, transform=PC, rasterized=True, zorder=2)
    cb = fig.colorbar(art, ax=ax, shrink=0.75, ticks=range(len(nombres_q)))
    cb.set_ticklabels([f"≥ P{k[1:]}" for k in nombres_q])
ax = axes[1, 3]; panel(ax, "Regiones de atribución")
reg = np.where(m_dir, 2.0, np.where(m_ind, 1.0, np.nan))
ax.pcolormesh(lon, lat, reg, cmap=mcolors.ListedColormap(["#9ecae1", "#2171b5"]),
              norm=mcolors.BoundaryNorm([0.5, 1.5, 2.5], 2), transform=PC, rasterized=True, zorder=2)
ax.legend(handles=[Patch(color="#2171b5", label="Directa (R34)"), Patch(color="#9ecae1", label="Indirecta (ROCLOUD, piso R_W06)")],
          loc="lower left", fontsize=7, frameon=True)

for ax, var, nom in zip(axes[2, :3], ("H_P", "H_W", "ITCHI"), ("$H_P$ (lluvia)", "$H_W$ (viento)", "ITCHI")):
    panel(ax, nom)
    art = plot_itchi_field(np.where(s[var].values > 0, s[var].values, np.nan), ax, lon=lon, lat=lat, crs=PC,
                           add_colorbar=False, rasterized=True, zorder=2)
    regiones(ax, lon, lat, m_dir, m_ind)
fig.colorbar(art, ax=axes[2, :3], shrink=0.75, label="Peligro [0, 1]")
ax = axes[2, 3]; panel(ax, "Clase SIAT de ITCHI")
plot_itchi_siat_levels(sin_clase_cero(s["ITCHI"].values), ax, lon=lon, lat=lat, crs=PC, add_colorbar=False,
                       rasterized=True, zorder=2)
regiones(ax, lon, lat, m_dir, m_ind)
barra_siat(fig, axes[2, 3], shrink=0.75)
fig.suptitle(f"{titulo(CASO_ESQ)} · {t:%Y-%m-%d %H} UTC · {float(s['track_vmax_kt']):.0f} kt "
             f"({categoria(float(s['track_vmax_kt']))})")
guardar(fig, f"fig00a_esquema_{CASO_ESQ.lower()}")

## Figura 0b · Rampas
(a) $H_P$ contra P6 en una celda real (la de mayor ITCHI sobre tierra en la posición de 0a), con cuatro
combinaciones de persistencia (R24) y lluvia antecedente (A72); las curvas salen de la misma función que usa
ITCHI (`precipitation_hazard` con la matriz de `configs/default.yaml`). (b) $H_W$ contra el viento local en
todas las posiciones del caso, con los nodos Saffir-Simpson y el corte de la cola. (c) Unión de los dos
componentes; la celda verifica con los datos que ITCHI = 1 − (1 − $H_P$)(1 − $H_W$).

In [ ]:
land = Cc["land"]
iy, ix = np.unravel_index(np.nanargmax(np.where(land, s["ITCHI"].values, -1)), land.shape)
q = {h: {k: float(v[iy, ix]) for k, v in Q[h].items()} for h in (6, 24, 72)}
print(f"celda {lat[iy]:.2f} N, {lon[ix]:.2f} E | P6:", {k: round(v, 1) for k, v in q[6].items()},
      "| R24:", {k: round(v, 1) for k, v in q[24].items()}, "| A72:", {k: round(v, 1) for k, v in q[72].items()})

p = np.linspace(0, 1.3 * max(q[6].values()), 400)
full = lambda d: {k: np.full_like(p, v) for k, v in d.items()}
bajo24, alto24 = 0.5 * q[24]["Q90"], q[24]["Q95"]
bajo72, alto72 = 0.0, q[72]["Q75"]
escenarios = {"R24 < P90, A72 = 0": (bajo24, bajo72), f"R24 ≥ P95 (sin A72)": (alto24, bajo72),
              f"A72 ≥ P75 (sin R24)": (bajo24, alto72), "R24 ≥ P95 y A72 ≥ P75": (alto24, alto72)}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), layout="constrained")
ax = axes[0]
for (nom, (r24, a72)), ls in zip(escenarios.items(), ("-", "--", ":", "-")):
    hp = precipitation_hazard(p, np.full_like(p, r24), np.full_like(p, a72), full(q[6]), full(q[24]), full(q[72]),
                              spec=SET["precip_matrix"], nan_value=SET["precipitation_nan_value"])
    ax.plot(p, hp, ls, lw=2.2 if nom.startswith("R24 ≥ P95 y") else 1.4, label=nom)
for k, v in q[6].items():
    ax.axvline(v, color="0.5", lw=0.7); ax.text(v, 1.02, k.replace("Q", "P"), ha="center", fontsize=7)
ax.set(xlabel="P6 (mm en 6 h)", ylabel="$H_P$", ylim=(-0.02, 1.08), title="(a) Lluvia: matriz P6 × R24 × A72")
ax.legend(fontsize=7, loc="lower right")

ax = axes[1]
ds_ = Cc["ds"]
reg_all = (ds_["M_direct"] | ds_["M_indirect"]).values.astype(bool)
v_all, hw_all = ds_["wind_kt"].values[reg_all], ds_["H_W"].sel(exp=EXP).values[reg_all]
sel = np.random.default_rng(0).choice(v_all.size, min(20000, v_all.size), replace=False)
ax.scatter(v_all[sel], hw_all[sel], s=2, color="0.3", alpha=0.3, rasterized=True, label="celdas del caso")
nodos = np.asarray(SAFFIR_SIMPSON_HAZARD_NODES, float)
ax.plot(nodos[:, 0], nodos[:, 1], "o", color="C3", label="nodos Saffir-Simpson")
for kt, nom in zip(nodos[:, 0], ("TT", "H1", "H2", "H3", "H5")):
    ax.text(kt, 1.03, nom, ha="center", fontsize=7)
ax.axvline(TAIL_END_WIND_MS * KT_POR_MS, color="C0", ls="--", lw=1, label=f"fin de la cola ({TAIL_END_WIND_MS:g} m/s)")
ax.set(xlabel="Viento local W06 (kt)", ylabel="$H_W$", ylim=(-0.02, 1.08), title="(b) Viento: escala absoluta")
ax.legend(fontsize=7, loc="lower right")

ax = axes[2]
g = np.linspace(0, 1, 201); HP, HW = np.meshgrid(g, g)
U = 1 - (1 - HP) * (1 - HW)
cmap, norm = get_siat_hazard_cmap_norm()
ax.contourf(HP, HW, classify_itchi_to_siat_levels(U), levels=np.arange(-0.5, 6), cmap=cmap, norm=norm, alpha=0.6)
cs = ax.contour(HP, HW, U, levels=[0.2, 0.4, 0.6, 0.8], colors="k", linewidths=0.6)
ax.clabel(cs, fontsize=7)
hp_s, hw_s = s["H_P"].values[m_dir | m_ind], s["H_W"].values[m_dir | m_ind]
ax.scatter(hp_s, hw_s, s=3, color="k", alpha=0.3, rasterized=True)
res = np.abs(ds_["ITCHI"].sel(exp=EXP).values - (1 - (1 - ds_["H_P"].sel(exp=EXP).values) * (1 - ds_["H_W"].sel(exp=EXP).values)))[reg_all]
ax.set(xlabel="$H_P$", ylabel="$H_W$", aspect="equal",
       title=f"(c) ITCHI = 1 − (1 − $H_P$)(1 − $H_W$)\nmáx. |ITCHI − unión| = {res.max():.1e}")
print(f"verificación de la unión: máx {res.max():.2e}, fracción > 1e-4: {(res > 1e-4).mean():.3%}")
guardar(fig, f"fig00b_rampas_{CASO_ESQ.lower()}")

## Figura 1 · Máximo del evento en la malla

In [ ]:
style = MapStyle(lon_tick_spacing=10.0, lat_tick_spacing=5.0, grid_spacing=5.0, label_size=8.0,
                 coastline_linewidth=0.7, border_linewidth=0.6, spine_linewidth=1.0)
fig, axes = plt.subplots(4, 2, figsize=(10, 12.5), subplot_kw={"projection": PC}, layout="constrained")
for ax, case in zip(axes.flat, CASOS):
    ds = C[case]["ds"]
    evmax = ds["ITCHI"].sel(exp=EXP).max("time").values
    configure_map_axis(ax, crs=PC, style=style)
    plot_itchi_siat_levels(sin_clase_cero(evmax, C[case]["land"] if MASCARA_TIERRA else None), ax,
                           lon=ds["lon"].values, lat=ds["lat"].values, crs=PC, add_colorbar=False,
                           rasterized=True, zorder=2)
    trayectoria(ax, C[case]["track"], (ds["track_lon"], ds["track_lat"]))
    ax.set_title(titulo(case))
axes.flat[-1].set_visible(False)
barra_siat(fig, axes[:, :], shrink=0.5, aspect=25)
guardar(fig, "fig01_itchi_max_malla")

## Figura 2 · Clase municipal y declaratorias

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(10, 12.5), subplot_kw={"projection": PC}, layout="constrained")
for ax, case in zip(axes.flat, CASOS):
    Cc, ds = C[case], C[case]["ds"]
    mun = municipal_event(Cc).sel(exp=EXP).max("time").values
    grid = paint_municipal(mun, Cc["pix"], Cc["codes"], (ds.sizes["lat"], ds.sizes["lon"]))
    lo, la = recuadro(ds["lon"].values, ds["lat"].values, (grid > 0) & Cc["land"])
    configure_map_axis(ax, crs=PC, style=estilo(lo, la))
    plot_itchi_siat_levels(sin_clase_cero(grid), ax, lon=ds["lon"].values, lat=ds["lat"].values, crs=PC,
                           add_colorbar=False, rasterized=True, zorder=2)
    g = Cc["gdf"]; sel = g[g["CVEGEO"].isin(DECL[case])]
    if len(sel):
        ax.add_geometries(sel.geometry, crs=PC, facecolor="none", edgecolor="k", linewidth=0.6, zorder=5)
    trayectoria(ax, Cc["track"], (ds["track_lon"], ds["track_lat"]))
    ax.set_title(f"{titulo(case)} · {len(sel)} con declaratoria")
axes.flat[-1].set_visible(False)
barra_siat(fig, axes[:, :], shrink=0.5, aspect=25)
fig.legend(handles=[Line2D([], [], color="k", lw=0.8, label="Municipio con declaratoria")],
           loc="lower right", bbox_to_anchor=(0.92, 0.1), frameon=False)
guardar(fig, "fig02_itchi_municipal_declaratorias")

## Figura 3 · Componentes en el pico del evento

In [ ]:
CASO_PICO = "CRISTOBAL2020"
Cc, ds = C[CASO_PICO], C[CASO_PICO]["ds"]
tp = peak_time(Cc, EXP)
snap = ds.sel(time=tp, exp=EXP)
lo, la = recuadro(ds["lon"].values, ds["lat"].values, (ds["ITCHI"].sel(exp=EXP).max("time").values > 0) & Cc["land"], margen=3.0)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), subplot_kw={"projection": PC}, layout="constrained")
for ax, var, nombre in zip(axes, ("H_P", "H_W", "ITCHI"), ("$H_P$ (lluvia)", "$H_W$ (viento)", "ITCHI")):
    configure_map_axis(ax, crs=PC, style=estilo(lo, la))
    art = plot_itchi_field(np.where(snap[var].values > 0, snap[var].values, np.nan), ax,
                           lon=ds["lon"].values, lat=ds["lat"].values, crs=PC, add_colorbar=False,
                           rasterized=True, zorder=2)
    regiones(ax, ds["lon"].values, ds["lat"].values, snap["M_direct"].values.astype(bool), snap["M_indirect"].values.astype(bool))
    ax.plot(float(snap["track_lon"]), float(snap["track_lat"]), "*", ms=10, color="k", transform=PC, zorder=7)
    ax.set_title(nombre)
fig.colorbar(art, ax=axes, shrink=0.8, label="Peligro normalizado [0, 1]")
fig.suptitle(f"{titulo(CASO_PICO)} · {pd.Timestamp(tp):%Y-%m-%d %H} UTC")
guardar(fig, f"fig03_componentes_pico_{CASO_PICO.lower()}")

## Figura 4 · Antes de los deslizamientos
Arriba: ITCHI municipal en las posiciones del ciclón y $H_P$ municipal continuo; cada valor cubre las 6 h que
terminan en su tiempo y los huecos sin posición del ciclón quedan en blanco. Abajo: percentil local más alto
alcanzado en el municipio por P6, R24 y A72; la escalada de la matriz pide R24 ≥ P95 y A72 ≥ P75.
Franja gris: hora del evento ± incertidumbre.

In [ ]:
VENTANA_ANTES_H, VENTANA_DESPUES_H = 120, 24
casos_d = [c for c in DESLIZAMIENTOS if c in C]
fig = plt.figure(figsize=(10, 4.6 * len(casos_d)), layout="constrained")
sub = fig.subfigures(len(casos_d), 1)
for sf, case in zip(np.atleast_1d(sub), casos_d):
    Cc, ds, d = C[case], C[case]["ds"], DESLIZAMIENTOS[case]
    cv = find_municipio(Cc["gdf"], d["municipio"], d["cve_ent"], d.get("cvegeo"))
    te, dh = t_event_utc(case), pd.Timedelta(hours=d["incertidumbre_h"])
    ax1, ax2 = sf.subplots(2, 1, sharex=True, gridspec_kw={"height_ratios": [2, 1.4]})
    t_all = pd.DatetimeIndex(ds["time_all"].values)

    for k in range(1, 6):
        ax1.axhspan(0.2 * (k - 1), 0.2 * k, color=SIAT_DANGER_COLORS[k], alpha=0.12, lw=0)
    hp_m = municipal_values(ds["H_P_cont"].sel(exp=EXP).values, Cc["pix"], [cv])[0]
    ax1.step(*con_huecos(t_all, hp_m), where="pre", color="0.45", lw=1.0, label="$H_P$ municipal (6 h)")
    itchi_m = municipal_event(Cc).sel(exp=EXP, mun=cv)
    ax1.step(*con_huecos(itchi_m["time"].values, itchi_m.values), where="pre", color="k", lw=1.8,
             label="ITCHI municipal (con posición del ciclón)")
    ax1.set(ylim=(0, 1), ylabel="Peligro [0, 1]", title=f"{titulo(case)} · {d['municipio']} ({cv})")

    nombres = sorted(set().union(*(cuantiles(ds, h) for h in (6, 24, 72))), key=lambda k: float(k[1:]))
    yv = {0.0: 0, **{float(k[1:]): i + 1 for i, k in enumerate(nombres)}}
    for (var, h), color in zip((("P6", 6), ("R24", 24), ("A72", 72)), ("C0", "C1", "C2")):
        lv = percentile_level(ds[var].values, cuantiles(ds, h))
        lv_m = municipal_values(lv, Cc["pix"], [cv])[0]
        ax2.step(t_all, [yv.get(x, np.nan) for x in lv_m], where="pre", color=color, lw=1.4, label=var)
    ax2.set_yticks(list(yv.values()), ["< P" + nombres[0][1:]] + [f"≥ P{k[1:]}" for k in nombres])
    ax2.set_ylabel("Percentil local\n(máx. del municipio)")
    ax2.legend(fontsize=7, ncol=3, loc="upper left", frameon=False)

    for ax in (ax1, ax2):
        ax.axvspan(te - dh, te + dh, color="0.6", alpha=0.5, lw=0)
        ax.axvline(te, color="k", ls="--", lw=1.0)
    ax1.legend(handles=ax1.get_legend_handles_labels()[0] + [Line2D([], [], color="k", ls="--", label="Deslizamiento")],
               fontsize=7, loc="upper left", frameon=False)
    ax2.set_xlim(te - pd.Timedelta(hours=VENTANA_ANTES_H), te + pd.Timedelta(hours=VENTANA_DESPUES_H))
    for lab in ax2.get_xticklabels():
        lab.set_rotation(30); lab.set_ha("right")
guardar(fig, "fig04_anticipacion_deslizamientos")

## Figura 5 · Evolución cada 6 h
ITCHI en cada posición de la trayectoria completa (también las que no alcanzan México, que no están en el
caché de `casos.py`; se calculan aquí y se guardan como `<caso>_completa.nc`). Sin máscara de tierra para ver
la estructura del ciclón; contornos: región directa (continuo) y de atribución (discontinuo).

In [ ]:
CASO_EVOL = "PATRICIA2015"
INICIO, PASO, PANELES = 0, 1, None             # PASO = 1: cada 6 h; PANELES = None: hasta el final
COLUMNAS = 4


def campos_completos(case):
    f = Path(cfg.cache_dir) / f"{case.lower()}_completa.nc"
    if f.exists():
        return xr.load_dataset(f)
    tr = load_track(case, cfg)
    p6, r24, a72 = read_precip(pd.DatetimeIndex(tr["time"]), cfg)
    qs = read_climatologies(p6.isel(time=0, drop=True), cfg)
    out = compute_case_fields(tr, p6, r24, a72, *qs, experiment_settings(cfg), verbose=True)
    out.to_netcdf(f, encoding={v: {"zlib": True, "complevel": 2} for v in out.data_vars})
    return out


dse = campos_completos(CASO_EVOL)
tiempos = pd.DatetimeIndex(dse["time"].values)[INICIO::PASO][:PANELES]
lon, lat = dse["lon"].values, dse["lat"].values
tr = C[CASO_EVOL]["track"]
lo = (max(tr["lon"].min() - 6, lon.min()), min(tr["lon"].max() + 6, lon.max()))
la = (max(tr["lat"].min() - 5, lat.min()), min(tr["lat"].max() + 5, lat.max()))
filas = int(np.ceil(len(tiempos) / COLUMNAS))
fig, axes = plt.subplots(filas, COLUMNAS, figsize=(3.6 * COLUMNAS, 3.0 * filas), subplot_kw={"projection": PC},
                         layout="constrained", squeeze=False)
for ax, tt in zip(axes.flat, tiempos):
    s_ = dse.sel(time=tt, exp=EXP)
    configure_map_axis(ax, crs=PC, style=estilo(lo, la, label_size=7.0))
    plot_itchi_siat_levels(sin_clase_cero(s_["ITCHI"].values), ax, lon=lon, lat=lat, crs=PC, add_colorbar=False,
                           rasterized=True, zorder=2)
    regiones(ax, lon, lat, s_["M_direct"].values.astype(bool), s_["M_indirect"].values.astype(bool))
    ax.plot(tr["lon"], tr["lat"], "-", color="0.4", lw=0.7, transform=PC, zorder=6)
    ax.plot(float(s_["track_lon"]), float(s_["track_lat"]), "*", ms=8, color="k", transform=PC, zorder=8)
    vmax = float(s_["track_vmax_kt"])
    ax.set_title(f"{tt:%d %b %H} UTC · {vmax:.0f} kt ({categoria(vmax)})", fontsize=8)
for ax in list(axes.flat)[len(tiempos):]:
    ax.set_visible(False)
barra_siat(fig, axes[:, :], shrink=0.6, aspect=30)
fig.suptitle(f"{titulo(CASO_EVOL)} · ITCHI cada 6 h")
guardar(fig, f"fig05_evolucion_{CASO_EVOL.lower()}")

## Tablas

In [ ]:
cc = pd.concat({titulo(c): class_counts(C[c], [EXP]) for c in CASOS}, names=["caso"])
ordn = pd.concat([tabla_ordinal(C[c], DECL[c], [EXP]).assign(caso=titulo(c)) for c in CASOS], ignore_index=True)
ordn_tot = ordn.groupby("clase")[["municipios", "con_declaratoria"]].sum()
ordn_tot["P_D"] = ordn_tot["con_declaratoria"] / ordn_tot["municipios"]
cc.to_csv(FIG_DIR / "tabla_clases_casos.csv"); ordn.to_csv(FIG_DIR / "tabla_ordinal_casos.csv", index=False)
display(cc)
display(ordn_tot.round(3))